In [1]:
# =========================
# SELECT MODEL
# =========================
MODEL_NAME = "efficientnet"   # "mobilenet" / "resnet" / "efficientnet"

# =========================
# SETUP
# =========================
import os, warnings
warnings.filterwarnings('ignore')

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import pandas as pd

print("GPUs:", tf.config.list_physical_devices('GPU'))

# =========================
# MODEL SELECTION
# =========================
if MODEL_NAME == "mobilenet":
    from tensorflow.keras.applications import MobileNetV2 as ModelClass
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

elif MODEL_NAME == "resnet":
    from tensorflow.keras.applications import ResNet50 as ModelClass
    from tensorflow.keras.applications.resnet50 import preprocess_input

elif MODEL_NAME == "efficientnet":
    from tensorflow.keras.applications import EfficientNetB0 as ModelClass
    from tensorflow.keras.applications.efficientnet import preprocess_input

# =========================
# CONFIG
# =========================
IMG_SIZE = (224, 224)
# IMG_SIZE = (160, 160)
BATCH_SIZE = 64    # 32
EPOCHS = 5    #15

TRAIN_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train"
TEST_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final"

# =========================
# DATA GENERATOR
# =========================
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# =========================
# LOAD MODEL (NO INTERNET ERROR)
# =========================
try:
    base_model = ModelClass(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
except:
    print("⚠️ Using random weights (no internet)")
    base_model = ModelClass(
        weights=None,
        include_top=False,
        input_shape=(224, 224, 3)
    )

# Freeze base
base_model.trainable = False

# =========================
# CUSTOM HEAD (ANTI-OVERFITTING)
# =========================
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.6)(x)

x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================
# CALLBACKS (ANTI-OVERFITTING)
# =========================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ModelCheckpoint("best_model.h5", monitor='val_loss', save_best_only=True)
]

# =========================
# TRAIN PHASE 1
# =========================
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# =========================
# FINE-TUNE
# =========================
# for layer in base_model.layers[-30:]:
#     layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

# =========================
# TEST DATA
# =========================
test_files = os.listdir(TEST_DIR)

test_df = pd.DataFrame({'filename': test_files})

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=TEST_DIR,
    x_col='filename',
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# =========================
# PREDICT
# =========================
predictions = model.predict(test_generator, verbose=0)

pred_labels = (predictions > 0.5).astype(int).reshape(-1)

label_map = {0: "chihuahua", 1: "muffin"}
pred_names = [label_map[x] for x in pred_labels]

submission = pd.DataFrame({
    'ID': test_df['filename'],
    'Label': pred_names
})

submission.to_csv("submission.csv", index=False)

print("✅ DONE - Submission created!")

E0000 00:00:1776593090.337378      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776593090.410885      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776593090.987930      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776593090.987979      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776593090.987982      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776593090.987985      55 computation_placer.cc:177] computation placer already registered. Please check linka

GPUs: []
Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.
⚠️ Using random weights (no internet)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ normalization_1[… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         64 │ block1a_project_

 Total params: 4,399,140 (16.78 MB)

...

60/60 ━━━━━━━━━━━━━━━━━━━━ 229s 4s/step - accuracy: 0.5373 - loss: 0.6904 - val_accuracy: 0.5407 - val_loss: 0.6895 - learning_rate: 3.0000e-06
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5456 - loss: 0.6888

60/60 ━━━━━━━━━━━━━━━━━━━━ 230s 4s/step - accuracy: 0.5455 - loss: 0.6888 - val_accuracy: 0.5407 - val_loss: 0.6895 - learning_rate: 3.0000e-06
Epoch 8/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5481 - loss: 0.6889

60/60 ━━━━━━━━━━━━━━━━━━━━ 229s 4s/step - accuracy: 0.5480 - loss: 0.6889 - val_accuracy: 0.5407 - val_loss: 0.6894 - learning_rate: 3.0000e-06
Epoch 9/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5387 - loss: 0.6893

60/60 ━━━━━━━━━━━━━━━━━━━━ 231s 4s/step - accuracy: 0.5388 - loss: 0.6893 - val_accuracy: 0.5407 - val_loss: 0.6894 - learning_rate: 1.0000e-06
Epoch 10/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5345 - loss: 0.6904

60/60 ━━━━━━━━━━━━━━━━━━━━ 267s 4s/step - accuracy: 0.5346 - loss: 0.6904 - val_accuracy: 0.5407 - val_loss: 0.6894 - learning_rate: 1.0000e-06
Found 1138 validated image filenames.
✅ DONE - Submission created!


In [3]:
import matplotlib.pyplot as plt

def plot_history(history):
    plt.figure(figsize=(12,5))

    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.legend()
    plt.title("Accuracy")

    # Loss
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.legend()
    plt.title("Loss")

    plt.show()

plot_history(history)

NameError: name 'history' is not defined

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# get predictions on validation
val_preds = model.predict(val_generator)
val_labels = val_generator.classes

val_preds = (val_preds > 0.5).astype(int)

cm = confusion_matrix(val_labels, val_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["chihuahua","muffin"])
disp.plot(cmap='Blues')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class_names = ["chihuahua", "muffin"]

def show_predictions(generator, model, n=6):
    images, labels = next(generator)
    preds = model.predict(images)

    plt.figure(figsize=(12,8))
    
    for i in range(n):
        plt.subplot(2,3,i+1)
        plt.imshow(images[i])
        
        pred_label = class_names[int(preds[i] > 0.5)]
        true_label = class_names[int(labels[i])]
        
        plt.title(f"Pred: {pred_label}\nTrue: {true_label}")
        plt.axis('off')
    
    plt.show()

show_predictions(val_generator, model)

In [ ]:
pred_probs = model.predict(val_generator)

plt.hist(pred_probs, bins=20)
plt.title("Prediction Confidence")
plt.show()

In [ ]:
def show_wrong_predictions(generator, model, n=6):
    images, labels = next(generator)
    preds = model.predict(images)

    wrong = np.where((preds > 0.5).astype(int).flatten() != labels)[0]

    plt.figure(figsize=(12,8))
    
    for i, idx in enumerate(wrong[:n]):
        plt.subplot(2,3,i+1)
        plt.imshow(images[idx])
        
        pred_label = class_names[int(preds[idx] > 0.5)]
        true_label = class_names[int(labels[idx])]
        
        plt.title(f"❌ Pred: {pred_label}\nTrue: {true_label}")
        plt.axis('off')
    
    plt.show()

show_wrong_predictions(val_generator, model)

In [ ]:
plot_history(history)
show_predictions(val_generator, model)
show_wrong_predictions(val_generator, model)

👉 Epochs are slow because the model is learning step-by-step from data

But yes—you can make it much faster without losing much accuracy. Let’s fix that practically.

🚨 First — reality check

You cannot completely remove epochs, but you can:

⚡ Reduce training time 3–10x
⚡ Get good results in few epochs
⚡ Stop early automatically
🚀 BEST WAYS TO TRAIN FASTER (KEEP SAME RESULT)
✅ 1. Reduce epochs + use EarlyStopping (already added)

Change:

EPOCHS = 15

👉 to:

EPOCHS = 5

👉 Because:

EarlyStopping will stop automatically anyway
No need to run all 15 epochs
✅ 2. Increase batch size (VERY IMPORTANT)

Change:

BATCH_SIZE = 32

👉 to:

BATCH_SIZE = 64

👉 Why:

Uses GPU better
Fewer steps per epoch → faster training
✅ 3. Train only head (fastest trick)

👉 Skip fine-tuning initially

Comment this part:

# for layer in base_model.layers[-30:]:
#     layer.trainable = True

👉 Result:

⚡ 2–5x faster
Still good accuracy
✅ 4. Use smaller model

Instead of:

MODEL_NAME = "efficientnet"

👉 Use:

MODEL_NAME = "mobilenet"

👉 Why:

MobileNet is lightweight
Much faster on T4 GPU
✅ 5. Reduce image size (huge speed boost)

Change:

IMG_SIZE = (224, 224)

👉 to:

IMG_SIZE = (160, 160)

👉 Result:

⚡ 30–50% faster
Slight accuracy drop (often acceptable)
✅ 6. Use fewer training samples (optional)
steps_per_epoch=50
validation_steps=20

👉 Add in model.fit()

⚡ FAST VERSION (recommended setup)

Use this combo:

MODEL_NAME = "mobilenet"
IMG_SIZE = (160,160)
BATCH_SIZE = 64
EPOCHS = 5

👉 This is the fastest + still good accuracy setup

🧠 Why training feels slow

Each epoch =

👉 forward pass + backward pass + weight update
👉 repeated for every image

So:

More images + bigger model = slower

🔥 Pro trick (BEST)

If you want very fast + decent results, do:

👉 Phase 1 only (skip fine-tune)
model.fit(...)

DONE.

👉 Don’t unfreeze layers.

🎯 Final recommendation
Goal	Setting
Fastest	MobileNet + small image
Balanced	ResNet
Best accuracy	EfficientNet
👍 Simple answer

👉 Yes, epochs take time
👉 But you can reduce time by:

smaller model
fewer epochs
bigger batch size
skipping fine-tuning